# U decomposition analysis

Generate Haar-random `SU(n)` matrices, decompose them with `unitary_to_G_rotations`, and convert the returned two-level rotations into ion-pulse parameters.

Notes:
- `mode="target"` converts the returned elimination rotations into the pulse sequence that synthesizes the target `U`.
- Each step returns `coupling`, `theta`, `phi`, and an extra diagonal phase `gamma`. `gamma` is the additional addressed-pair phase that must be tracked explicitly.
- For the `haar_su` sampler below, the residual `V` is numerically the identity. For a more general `U(n)` input, a residual diagonal can remain.
- Set `n = 29` when you want the active Ba-137 manifold size instead of a small test case.


In [2]:
import numpy as np
from numpy.linalg import norm

from U_decomp import unitary_to_G_rotations

np.set_printoptions(precision=4, suppress=True)


In [3]:
def haar_unitary(n, rng=None):
    rng = np.random.default_rng(rng)
    X = (rng.standard_normal((n, n)) + 1j * rng.standard_normal((n, n))) / np.sqrt(2)
    Q, R = np.linalg.qr(X)
    d = np.diag(R)
    D = d / np.abs(d)
    return Q * D


def haar_su(n, rng=None):
    U = haar_unitary(n, rng)
    phi = np.angle(np.linalg.det(U)) / n
    return U * np.exp(-1j * phi)


import numpy as np
from collections import deque
from math import gcd

# ============================================================
# Single-qudit Clifford group generator
# Supports:
#   - d = 2   (qubit)
#   - odd prime d = 3, 5, 7, ...
#
# It returns Clifford unitaries modulo global phase.
# ============================================================

def is_prime(n: int) -> bool:
    if n < 2:
        return False
    if n % 2 == 0:
        return n == 2
    p = 3
    while p * p <= n:
        if n % p == 0:
            return False
        p += 2
    return True


def modinv(a: int, m: int) -> int:
    """Modular inverse of a mod m."""
    a %= m
    for x in range(1, m):
        if (a * x) % m == 1:
            return x
    raise ValueError(f"No modular inverse for {a} mod {m}")


def canonicalize_global_phase(U: np.ndarray, tol: float = 1e-10) -> np.ndarray:
    """
    Fix a canonical global phase so matrices that differ only by a global
    phase compare equal numerically.
    """
    U = np.array(U, dtype=complex)

    # Find first entry with non-negligible magnitude
    idx = None
    flat = U.flatten()
    for k, z in enumerate(flat):
        if abs(z) > tol:
            idx = k
            break

    if idx is None:
        raise ValueError("Zero matrix cannot be canonicalized.")

    phase = flat[idx] / abs(flat[idx])
    U = U / phase

    # Clean tiny numerical noise
    U.real[abs(U.real) < tol] = 0.0
    U.imag[abs(U.imag) < tol] = 0.0
    return U


def matrix_key(U: np.ndarray, decimals: int = 10) -> tuple:
    """
    Hashable key for a unitary modulo global phase.
    """
    Uc = canonicalize_global_phase(U)
    return tuple(np.round(Uc.flatten(), decimals=decimals))


def dft_matrix(d: int) -> np.ndarray:
    """Generalized Fourier transform F_d."""
    omega = np.exp(2j * np.pi / d)
    j, k = np.meshgrid(np.arange(d), np.arange(d), indexing="ij")
    return omega ** (j * k) / np.sqrt(d)


def qubit_generators():
    """
    Standard qubit generators H and S.
    These generate the single-qubit Clifford group modulo phase.
    """
    H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
    S = np.array([[1, 0], [0, 1j]], dtype=complex)
    return [H, S]


def odd_prime_qudit_generators(d: int):
    """
    Generators for single-qudit Clifford group in odd prime dimension d:
      F = Fourier transform
      P = quadratic phase gate with diagonal omega^{j(j-1)/2}
    """
    omega = np.exp(2j * np.pi / d)
    F = dft_matrix(d)

    inv2 = modinv(2, d)
    diag = [omega ** ((j * (j - 1) * inv2) % d) for j in range(d)]
    P = np.diag(diag)

    return [F, P]


def generalized_paulis(d: int):
    """
    Return X and Z for a single qudit of dimension d.
    Useful for verification.
    """
    omega = np.exp(2j * np.pi / d)

    X = np.zeros((d, d), dtype=complex)
    for j in range(d):
        X[(j + 1) % d, j] = 1

    Z = np.diag([omega ** j for j in range(d)])
    return X, Z


def generate_single_qudit_clifford_group(d: int, max_elements: int | None = None):
    """
    Generate the full single-qudit Clifford group modulo global phase.

    Parameters
    ----------
    d : int
        Dimension of the qudit.
        Supported: d = 2, or odd prime d.
    max_elements : int or None
        Optional safeguard to stop after reaching this many elements.

    Returns
    -------
    group : list[np.ndarray]
        List of unique Clifford matrices modulo global phase.
    """
    if d == 2:
        gens = qubit_generators()
    else:
        if not is_prime(d) or d % 2 == 0:
            raise ValueError(
                "This script supports d=2 or odd prime d only "
                "(e.g. 3, 5, 7, ...)."
            )
        gens = odd_prime_qudit_generators(d)

    # Include inverses too, so closure is reached faster
    gens = gens + [g.conj().T for g in gens]

    I = np.eye(d, dtype=complex)
    seen = {}
    q = deque()

    kI = matrix_key(I)
    seen[kI] = canonicalize_global_phase(I)
    q.append(seen[kI])

    while q:
        current = q.popleft()

        for g in gens:
            new = current @ g
            k = matrix_key(new)
            if k not in seen:
                seen[k] = canonicalize_global_phase(new)
                q.append(seen[k])

                if max_elements is not None and len(seen) >= max_elements:
                    return list(seen.values())

    return list(seen.values())


def verify_clifford(U: np.ndarray, d: int, tol: float = 1e-8) -> bool:
    """
    Basic verification: checks whether U X U† and U Z U† are generalized Pauli
    operators up to phase, by brute force over X^a Z^b for single-qudit case.
    """
    X, Z = generalized_paulis(d)

    paulis = []
    for a in range(d):
        Xa = np.linalg.matrix_power(X, a)
        for b in range(d):
            Zb = np.linalg.matrix_power(Z, b)
            paulis.append(Xa @ Zb)

    def matches_pauli(A):
        for P in paulis:
            # Compare up to global phase
            try:
                Ak = matrix_key(A)
                Pk = matrix_key(P)
                if Ak == Pk:
                    return True
            except Exception:
                pass
        return False

    Udag = U.conj().T
    return matches_pauli(U @ X @ Udag) and matches_pauli(U @ Z @ Udag)


if __name__ == "__main__":
    # Example 1: qubit
    d = 2
    G2 = generate_single_qudit_clifford_group(d)
    print(f"d={d}: generated {len(G2)} Cliffords modulo global phase")

    # Example 2: qutrit
    d = 3
    G3 = generate_single_qudit_clifford_group(d)
    print(f"d={d}: generated {len(G3)} Cliffords modulo global phase")

    # Verify a few elements
    print("Verification on first 5 qutrit elements:")
    for i, U in enumerate(G3[:5]):
        print(i, verify_clifford(U, 3))

    # Show one example matrix
    print("\nOne Clifford matrix for d=3:")
    print(G3[1])

d=2: generated 24 Cliffords modulo global phase
d=3: generated 216 Cliffords modulo global phase
Verification on first 5 qutrit elements:
0 True
1 True
2 True
3 True
4 True

One Clifford matrix for d=3:
[[ 0.5774+0.j   0.5774+0.j   0.5774+0.j ]
 [ 0.5774+0.j  -0.2887+0.5j -0.2887-0.5j]
 [ 0.5774+0.j  -0.2887-0.5j -0.2887+0.5j]]


In [4]:
def wrap_pi(x):
    return (x + np.pi) % (2 * np.pi) - np.pi


def ion_pulse_unitary(coupling, theta, phi, dim):
    i, j = coupling
    U = np.eye(dim, dtype=complex)
    c = np.cos(theta / 2)
    s = np.sin(theta / 2)
    U[i, i] = c
    U[j, j] = c
    U[i, j] = -1j * np.exp(1j * phi) * s
    U[j, i] = -1j * np.exp(-1j * phi) * s
    return U


def phase_update_unitary(coupling, gamma, dim):
    i, j = coupling
    D = np.eye(dim, dtype=complex)
    D[i, i] = np.exp(1j * gamma)
    D[j, j] = np.exp(-1j * gamma)
    return D


def rotation_matrix_to_pulse_parameters(G, tol=1e-10):
    G = np.asarray(G, dtype=complex)
    if G.ndim != 2 or G.shape[0] != G.shape[1]:
        raise ValueError("G must be square")

    dim = G.shape[0]
    candidates = [(p, q) for p in range(dim) for q in range(p + 1, dim)
                  if abs(G[p, q]) > tol or abs(G[q, p]) > tol]
    if len(candidates) != 1:
        raise ValueError(f"Expected exactly one coupled pair, found {candidates}")

    i, j = candidates[0]
    mask = np.ones_like(G, dtype=bool)
    mask[[i, j], :] = False
    mask[:, [i, j]] = False
    if norm(G[mask] - np.eye(dim, dtype=complex)[mask]) > 100 * tol:
        raise ValueError("Extra couplings detected outside the active 2x2 block")

    c = G[i, i]
    s = G[i, j]
    gamma = float(wrap_pi(np.angle(c)))
    theta = float(2.0 * np.arctan2(np.clip(np.abs(s), 0.0, 1.0), np.clip(np.abs(c), 0.0, 1.0)))
    phi = float((np.angle(s) - gamma + np.pi / 2) % (2 * np.pi))

    coupling = (i, j)
    reconstructed = phase_update_unitary(coupling, gamma, dim) @ ion_pulse_unitary(coupling, theta, phi, dim)

    return {
        "coupling": coupling,
        "theta": theta,
        "theta_over_pi": float(theta / np.pi),
        "phi": phi,
        "gamma": gamma,
        "reconstruction_error": float(norm(reconstructed - G)),
    }


def rotations_to_ion_schedule(rotation_mats, mode="target", tol=1e-10):
    if mode not in {"target", "elimination"}:
        raise ValueError("mode must be 'target' or 'elimination'")
    if len(rotation_mats) == 0:
        return []

    if mode == "target":
        sequence = [G.conj().T for G in rotation_mats[::-1]]
    else:
        sequence = list(rotation_mats)

    dim = sequence[0].shape[0]
    z_frame = np.zeros(dim, dtype=float)
    schedule = []

    for step_idx, G in enumerate(sequence, start=1):
        step = rotation_matrix_to_pulse_parameters(G, tol=tol)
        i, j = step["coupling"]
        gamma = step["gamma"]
        z_frame[i] += gamma
        z_frame[j] -= gamma

        step["step"] = step_idx
        step["z_phase_update"] = {i: gamma, j: -gamma}
        step["z_frame_after"] = z_frame.copy()
        schedule.append(step)

    return schedule


def schedule_to_lists(schedule):
    couplings = [step["coupling"] for step in schedule]
    thetas = [step["theta"] for step in schedule]
    phis = [step["phi"] for step in schedule]
    gammas = [step["gamma"] for step in schedule]
    return couplings, thetas, phis, gammas


def unitary_from_ion_schedule(couplings, thetas, phis, dim, gammas=None, return_z_frame=False):
    if not (len(couplings) == len(thetas) == len(phis)):
        raise ValueError("couplings, thetas, and phis must have the same length")
    if gammas is not None and len(gammas) != len(couplings):
        raise ValueError("gammas must match the number of pulses")

    U = np.eye(dim, dtype=complex)
    z_frame = np.zeros(dim, dtype=float)

    for idx, (coupling, theta, phi) in enumerate(zip(couplings, thetas, phis)):
        step_unitary = ion_pulse_unitary(coupling, theta, phi, dim)

        if gammas is not None:
            gamma = gammas[idx]
            step_unitary = phase_update_unitary(coupling, gamma, dim) @ step_unitary
            i, j = coupling
            z_frame[i] += gamma
            z_frame[j] -= gamma

        U = step_unitary @ U

    if return_z_frame:
        return U, z_frame
    return U


In [5]:
n = 3  # set to 29 for the active Ba-137 manifold
rng = 1234
center = 0

U = haar_su(n, rng=rng)

# Hadamard = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
# Hadamard_2 = np.kron(Hadamard, Hadamard)
# Hadamard_3 = np.kron(Hadamard_2, Hadamard)
# U = Hadamard_3 

# U = generate_single_qudit_clifford_group(n, max_elements=10)[3]
# print(U@U.conj().T)
rotation_mats, V = unitary_to_G_rotations(U, center=center)

schedule = rotations_to_ion_schedule(rotation_mats, mode="target")
couplings, thetas, phis, gammas = schedule_to_lists(schedule)

print(f"n = {n}, center = {center}")
print(f"number of TAQR rotations: {len(rotation_mats)}")
print(f"||V - I|| = {norm(V - np.eye(n)):.3e}")
print("\nTarget-side pulse schedule:")

for step in schedule:
    i, j = step["coupling"]
    gamma = step["gamma"]
    if abs(gamma) < 1e-12:
        phase_note = "no extra diagonal phase"
    else:
        phase_note = f"extra phase: level {i} += {gamma:+.6f}, level {j} += {-gamma:+.6f}"

    print(
        f"step {step['step']:2d}: coupling={step['coupling']}, "
        f"theta={step['theta']:.6f} rad ({step['theta_over_pi']:.6f} pi), "
        f"phi={step['phi']:.6f} rad, gamma={gamma:+.6f} rad, {phase_note}"
    )

print("\nCopy/paste lists:")
print("couplings =", couplings)
print("thetas    =", thetas)
print("phis      =", phis)
print("gammas    =", gammas)


n = 3, center = 0
number of TAQR rotations: 3
||V - I|| = 6.232e-16

Target-side pulse schedule:
step  1: coupling=(0, 1), theta=0.797701 rad (0.253916 pi), phi=5.715741 rad, gamma=+0.703227 rad, extra phase: level 0 += +0.703227, level 1 += -0.703227
step  2: coupling=(0, 2), theta=2.172033 rad (0.691380 pi), phi=4.904742 rad, gamma=+2.949239 rad, extra phase: level 0 += +2.949239, level 2 += -2.949239
step  3: coupling=(0, 1), theta=1.000672 rad (0.318524 pi), phi=6.004507 rad, gamma=-0.693536 rad, extra phase: level 0 += -0.693536, level 1 += +0.693536

Copy/paste lists:
couplings = [(0, 1), (0, 2), (0, 1)]
thetas    = [0.7977009566355381, 2.1720330983882232, 1.0006717099358724]
phis      = [5.715740500413014, 4.9047423508918175, 6.004506722945826]
gammas    = [0.7032267231588545, 2.9492392830826653, -0.693536287135506]


In [7]:
U_pulses_only = unitary_from_ion_schedule(couplings, thetas, phis, dim=n)
U_with_gammas, z_frame = unitary_from_ion_schedule(
    couplings, thetas, phis, dim=n, gammas=gammas, return_z_frame=True
)

print("max per-step reconstruction error:", max(step['reconstruction_error'] for step in schedule))
print("pulse-only error vs target U:", norm(U_pulses_only - U))
print('Phase only U:')
print(U_pulses_only)
print("pulse+gamma error vs target U:", norm(U_with_gammas - U))
print('Pulse+gamma U:')
print(U_with_gammas)
print("final accumulated z-frame:")
print(z_frame)


max per-step reconstruction error: 5.530259263078839e-16
pulse-only error vs target U: 2.688631454783582
Phase only U:
[[ 0.1982-0.0531j -0.207 -0.5589j -0.762 -0.1484j]
 [ 0.2398-0.4854j  0.7253+0.0247j -0.1926+0.3782j]
 [ 0.8003-0.1559j -0.2367-0.2491j  0.466 +0.j    ]]
pulse+gamma error vs target U: 8.919110737787415e-16
Pulse+gamma U:
[[-0.4537+0.2352j -0.3314+0.1624j  0.597 -0.4963j]
 [-0.0249-0.2711j  0.8614+0.061j   0.3506-0.2392j]
 [-0.6219-0.5273j -0.0465+0.3404j -0.4574-0.0891j]]
final accumulated z-frame:
[ 2.9589 -0.0097 -2.9492]


## Gamma minimization and virtual-Z compilation

Allowing `theta` to run from `0` to `2*pi` gives a second exact branch for every step:

- original branch: `(theta, phi, gamma)`
- alternate branch: `(2*pi - theta, phi + pi, wrap_pi(gamma + pi))`

That alternate branch cannot eliminate `gamma` in general, but it can reduce `|gamma|` to at most `pi/2` for each step.

After that, the remaining `gamma` values can be folded into a running virtual-Z frame. With the convention used here, the exact compiled unitary is

`U_target = D_final @ U_pulses`

where `U_pulses` is built from the programmed LO phases and `D_final = diag(exp(1j * z_final))` is the final virtual-Z frame.


In [8]:
def minimize_gamma_for_step(step):
    alternate = dict(step)
    alternate["theta"] = float(2 * np.pi - step["theta"])
    alternate["theta_over_pi"] = float(alternate["theta"] / np.pi)
    alternate["phi"] = float((step["phi"] + np.pi) % (2 * np.pi))
    alternate["gamma"] = float(wrap_pi(step["gamma"] + np.pi))
    return alternate if abs(alternate["gamma"]) < abs(step["gamma"]) else dict(step)


def minimize_schedule_gamma(schedule):
    return [minimize_gamma_for_step(step) for step in schedule]


def virtual_z_diagonal(z_frame):
    return np.diag(np.exp(1j * np.asarray(z_frame, dtype=float)))


def compile_virtual_z_schedule(schedule):
    if len(schedule) == 0:
        return [], np.array([])

    if "z_frame_after" in schedule[0]:
        dim = len(schedule[0]["z_frame_after"])
    else:
        dim = max(max(step["coupling"]) for step in schedule) + 1

    frame = np.zeros(dim, dtype=float)
    compiled = []

    for step in schedule:
        i, j = step["coupling"]
        programmed_phi = float((step["phi"] - (frame[i] - frame[j])) % (2 * np.pi))

        compiled_step = dict(step)
        compiled_step["frame_before"] = frame.copy()
        compiled_step["phi_programmed"] = programmed_phi

        frame[i] += step["gamma"]
        frame[j] -= step["gamma"]

        compiled_step["frame_after"] = frame.copy()
        compiled.append(compiled_step)

    return compiled, frame


def programmed_schedule_to_unitary(couplings, thetas, programmed_phis, dim, final_frame=None):
    U_pulses = unitary_from_ion_schedule(couplings, thetas, programmed_phis, dim)
    if final_frame is None:
        return U_pulses
    return virtual_z_diagonal(final_frame) @ U_pulses


In [9]:
schedule_min = minimize_schedule_gamma(schedule)
compiled_schedule, z_final = compile_virtual_z_schedule(schedule_min)

couplings_vz = [step["coupling"] for step in compiled_schedule]
thetas_vz = [step["theta"] for step in compiled_schedule]
phis_vz = [step["phi_programmed"] for step in compiled_schedule]
gammas_vz = [step["gamma"] for step in compiled_schedule]

print("max |gamma| before minimization:", max(abs(step["gamma"]) for step in schedule))
print("max |gamma| after minimization :", max(abs(step["gamma"]) for step in schedule_min))
print("\nVirtual-Z compiled schedule:")

couplings = []
thetas = []
phases = []

for step in compiled_schedule:
    print(
        f"step {step['step']:2d}: coupling={step['coupling']}, "
        f"theta={step['theta']:.6f} rad ({step['theta_over_pi']:.6f} pi), "
        f"phi_programmed={step['phi_programmed']:.6f} rad, gamma={step['gamma']:+.6f} rad"
    )
    couplings.append(step["coupling"])
    thetas.append(step["theta"])
    phases.append(step["phi_programmed"])

print("\nProgrammed LO phases:", phis_vz)
print("Final virtual-Z frame:")
print(z_final)

U_programmed_only = programmed_schedule_to_unitary(couplings_vz, thetas_vz, phis_vz, dim=n)
U_programmed_exact = programmed_schedule_to_unitary(couplings_vz, thetas_vz, phis_vz, dim=n, final_frame=z_final)

print("\nprogrammed-pulses-only error vs target U:", norm(U_programmed_only - U))
# print(U_programmed_only)
print("programmed-pulses + final virtual-Z error vs target U:", norm(U_programmed_exact - U))
print(U_programmed_exact)
print(U)


max |gamma| before minimization: 2.9492392830826653
max |gamma| after minimization : 0.7032267231588545

Virtual-Z compiled schedule:
step  1: coupling=(0, 1), theta=0.797701 rad (0.253916 pi), phi_programmed=5.715741 rad, gamma=+0.703227 rad
step  2: coupling=(0, 2), theta=4.111152 rad (1.308620 pi), phi_programmed=1.059923 rad, gamma=-0.192353 rad
step  3: coupling=(0, 1), theta=1.000672 rad (0.318524 pi), phi_programmed=4.790407 rad, gamma=-0.693536 rad

Programmed LO phases: [5.715740500413014, 1.05992297414317, 4.790406647135245]
Final virtual-Z frame:
[-0.1827 -0.0097  0.1924]

programmed-pulses-only error vs target U: 0.265052559589447
programmed-pulses + final virtual-Z error vs target U: 9.14487249829951e-16
[[-0.4537+0.2352j -0.3314+0.1624j  0.597 -0.4963j]
 [-0.0249-0.2711j  0.8614+0.061j   0.3506-0.2392j]
 [-0.6219-0.5273j -0.0465+0.3404j -0.4574-0.0891j]]
[[-0.4537+0.2352j -0.3314+0.1624j  0.597 -0.4963j]
 [-0.0249-0.2711j  0.8614+0.061j   0.3506-0.2392j]
 [-0.6219-0.5273j

In [76]:
print(sum(thetas)/np.pi)

2.3918265520306075
